# Dataset summary

Sanity checks and top-line descriptives for both analysis-ready datasets.
Template for every other notebook: import `lib`, load, explore.

In [ ]:
import sys; sys.path.append("..")
import polars as pl
from lib import data, codebooks, viz
viz.apply_style()

## Accusation dataset (532k, pandas)

In [ ]:
acc = data.load_accusations()
print(acc.shape)
acc[["country", "date", "target_type", "accuser_match", "target_match"]].head()

In [ ]:
# accuser resolution and target-type distribution
print(acc["accuser_match"].value_counts())
print()
print(acc["target_type"].map(codebooks.TARGET_TYPE).value_counts())

In [ ]:
# accuser ideology where resolved
res = data.resolved_accusers(acc)
res["accuser_left_right"].describe()

## Full corpus (149M, polars lazy)

Never load whole. Push aggregations down, collect only the small result.

In [ ]:
lf = data.scan_corpus()

# accusation rate by country (lie_score above threshold)
rate = (
    lf.with_columns(pl.col("lie_score").cast(pl.Float64))
      .group_by("country")
      .agg([
          pl.len().alias("n_sentences"),
          (pl.col("lie_score") > data.LIE_THRESHOLD).mean().alias("acc_rate"),
      ])
      .sort("n_sentences", descending=True)
      .collect()
)
rate